# 00 — Dataset Overview

High-level map of the **Home Credit Default Risk** relational dataset: table inventory, grains, join keys, train-client coverage in auxiliary tables, and train/test schema alignment.

## Purpose

This notebook answers dataset-level questions before table-specific EDA:

- Which CSV files exist and how large are they?
- What is the grain of each table (one row = what)?
- How do tables join via `SK_ID_CURR`, `SK_ID_PREV`, and `SK_ID_BUREAU`?
- What share of train clients have rows in bureau, previous applications, installments, etc.?
- Are `application_train` and `application_test` structurally aligned?

## Follow-up notebooks

| Notebook | Focus |
|----------|-------|
| `01_application_eda.ipynb` | Application table deep dive |
| `02_bureau_eda.ipynb` | Bureau + bureau_balance |
| `03_previous_applications_eda.ipynb` | Previous applications |
| `04_installments_eda.ipynb` | Installments payments |
| `05_pos_cash_eda.ipynb` | POS/cash balance |
| `06_credit_card_eda.ipynb` | Credit card balance |
| `07_protected_attributes.ipynb` | Gender, age, gender × age bands, proxy candidates |

## Out of scope here

- **Credit scoring model training**
- **Null-importance feature selection**
- **Fairness metrics / mitigation** 

High missingness or low coverage may be noted as implications for feature engineering, but not analyzed in depth here.

In [3]:
TABLE_SPECS = {
    "application_train": {
        "file": "application_train.csv",
        "grain": "One row per current loan application (client)",
        "keys": ["SK_ID_CURR"],
    },
    "application_test": {
        "file": "application_test.csv",
        "grain": "One row per current loan application (client)",
        "keys": ["SK_ID_CURR"],
    },
    "bureau": {
        "file": "bureau.csv",
        "grain": "One row per previous credit bureau credit",
        "keys": ["SK_ID_CURR", "SK_ID_BUREAU"],
    },
    "bureau_balance": {
        "file": "bureau_balance.csv",
        "grain": "One row per bureau credit per monthly status snapshot",
        "keys": ["SK_ID_BUREAU"],
    },
    "previous_application": {
        "file": "previous_application.csv",
        "grain": "One row per previous Home Credit application",
        "keys": ["SK_ID_CURR", "SK_ID_PREV"],
    },
    "POS_CASH_balance": {
        "file": "POS_CASH_balance.csv",
        "grain": "One row per previous application per monthly POS/cash balance snapshot",
        "keys": ["SK_ID_CURR", "SK_ID_PREV"],
    },
    "installments_payments": {
        "file": "installments_payments.csv",
        "grain": "One row per scheduled/actual installment payment",
        "keys": ["SK_ID_CURR", "SK_ID_PREV"],
    },
    "credit_card_balance": {
        "file": "credit_card_balance.csv",
        "grain": "One row per previous credit card application per monthly balance snapshot",
        "keys": ["SK_ID_CURR", "SK_ID_PREV"],
    },
    "HomeCredit_columns_description": {
        "file": "HomeCredit_columns_description.csv",
        "grain": "One row per documented column description",
        "keys": [],
    },
}

EXPECTED_FILES = [spec["file"] for spec in TABLE_SPECS.values()]


def check_data_files(data_dir: Path, repo_root: Path) -> pd.DataFrame:
    rows = []
    for name, spec in TABLE_SPECS.items():
        path = data_dir / spec["file"]
        try:
            rel_path = path.resolve().relative_to(repo_root.resolve())
        except ValueError:
            rel_path = path
        rows.append(
            {
                "table": name,
                "file": spec["file"],
                "exists": path.exists(),
                "path": str(rel_path),
            }
        )
    return pd.DataFrame(rows)


file_check = check_data_files(DATA_DIR, REPO_ROOT)
missing = file_check.loc[~file_check["exists"], "file"].tolist()

if missing:
    raise FileNotFoundError(
        "Missing expected CSV files in data/. Place Home Credit CSVs in "
        f"{DATA_DIR}. Missing: {missing}"
    )

---

## 1. CSV inventory: shape and grain

In [10]:
def load_tables(data_dir: Path, specs: dict) -> dict[str, pd.DataFrame]:
    tables = {}
    for name, spec in specs.items():
        path = data_dir / spec["file"]
        # print(f"Loading {spec['file']}...")
        read_kwargs = {}
        if name == "HomeCredit_columns_description":
            read_kwargs["encoding"] = "latin-1"
        tables[name] = pd.read_csv(path, **read_kwargs)
        # print(f"  -> {tables[name].shape[0]:,} rows x {tables[name].shape[1]} columns")
    return tables


tables = load_tables(DATA_DIR, TABLE_SPECS)

def summarize_tables(tables: dict, specs: dict) -> pd.DataFrame:
    rows = []
    for name, spec in specs.items():
        df = tables[name]
        present_keys = [k for k in spec["keys"] if k in df.columns]
        rows.append(
            {
                "table": name,
                "file": spec["file"],
                "rows": df.shape[0],
                "columns": df.shape[1],
                "key_columns": ", ".join(present_keys) if present_keys else "—",
                "grain": spec["grain"],
            }
        )
    summary = pd.DataFrame(rows)
    return summary.sort_values("rows", ascending=False).reset_index(drop=True)


inventory = summarize_tables(tables, TABLE_SPECS)
inventory

,table,file,rows,columns,key_columns,grain
0,bureau_balance,bureau_balance.csv,27299925,3,SK_ID_BUREAU,One row per bureau credit per monthly status snapshot
1,installments_payments,installments_payments.csv,13605401,8,"SK_ID_CURR, SK_ID_PREV",One row per scheduled/actual installment payment
2,POS_CASH_balance,POS_CASH_balance.csv,10001358,8,"SK_ID_CURR, SK_ID_PREV",One row per previous application per monthly POS/cash balance snapshot
3,credit_card_balance,credit_card_balance.csv,3840312,23,"SK_ID_CURR, SK_ID_PREV",One row per previous credit card application per monthly balance snapshot
4,bureau,bureau.csv,1716428,17,"SK_ID_CURR, SK_ID_BUREAU",One row per previous credit bureau credit
5,previous_application,previous_application.csv,1670214,37,"SK_ID_CURR, SK_ID_PREV",One row per previous Home Credit application
6,application_train,application_train.csv,307511,122,SK_ID_CURR,One row per current loan application (client)
7,application_test,application_test.csv,48744,121,SK_ID_CURR,One row per current loan application (client)
8,HomeCredit_columns_description,HomeCredit_columns_description.csv,219,5,—,One row per documented column description


---

## 2. Join keys and relationship diagram

Tables link at three identifiers:

| Key | Role |
|-----|------|
| `SK_ID_CURR` | Current loan application / client |
| `SK_ID_PREV` | Previous Home Credit application |
| `SK_ID_BUREAU` | Bureau credit record |

![Home Credit dataset relationships](../../home_credit.png)

In [5]:
RELATIONSHIPS = [
    {
        "parent": "application_train / application_test",
        "child": "bureau",
        "join_key": "SK_ID_CURR",
        "cardinality": "one-to-many",
    },
    {
        "parent": "bureau",
        "child": "bureau_balance",
        "join_key": "SK_ID_BUREAU",
        "cardinality": "one-to-many",
    },
    {
        "parent": "application_train / application_test",
        "child": "previous_application",
        "join_key": "SK_ID_CURR",
        "cardinality": "one-to-many",
    },
    {
        "parent": "previous_application",
        "child": "POS_CASH_balance",
        "join_key": "SK_ID_PREV",
        "cardinality": "one-to-many",
    },
    {
        "parent": "previous_application",
        "child": "installments_payments",
        "join_key": "SK_ID_PREV",
        "cardinality": "one-to-many",
    },
    {
        "parent": "previous_application",
        "child": "credit_card_balance",
        "join_key": "SK_ID_PREV",
        "cardinality": "one-to-many",
    },
]

relationships_df = pd.DataFrame(RELATIONSHIPS)
relationships_df

,parent,child,join_key,cardinality
0,application_train / application_test,bureau,SK_ID_CURR,one-to-many
1,bureau,bureau_balance,SK_ID_BUREAU,one-to-many
2,application_train / application_test,previous_application,SK_ID_CURR,one-to-many
3,previous_application,POS_CASH_balance,SK_ID_PREV,one-to-many
4,previous_application,installments_payments,SK_ID_PREV,one-to-many
5,previous_application,credit_card_balance,SK_ID_PREV,one-to-many


---

## 3. Coverage: train clients in auxiliary tables

For each auxiliary domain we report how many train clients have at least one related row, plus row-count multiplicity among covered clients.

In [15]:
def coverage_by_client(
    train_ids: pd.Series,
    aux_df: pd.DataFrame,
    client_key: str = "SK_ID_CURR",
    label: str | None = None,
) -> dict:
    train_set = set(train_ids)
    n_train = len(train_set)

    if client_key not in aux_df.columns:
        raise KeyError(f"{label or 'table'} missing column {client_key}")

    counts = aux_df.groupby(client_key).size()
    covered_ids = set(counts.index) & train_set
    n_covered = len(covered_ids)

    covered_counts = counts.loc[counts.index.isin(covered_ids)] if n_covered else pd.Series(dtype=float)

    return {
        "domain": label or client_key,
        "total_clients": n_train,
        "covered_clients": n_covered,
        "%coverage": round(100 * n_covered / n_train, 2) if n_train else 0.0,
        "rows_min": int(covered_counts.min()) if n_covered else 0,
        "rows_median": float(covered_counts.median()) if n_covered else 0.0,
        "rows_mean": round(float(covered_counts.mean()), 2) if n_covered else 0.0,
        "rows_max": int(covered_counts.max()) if n_covered else 0,
    }


train_ids = tables["application_train"]["SK_ID_CURR"]

coverage_rows = [
    coverage_by_client(train_ids, tables["bureau"], label="bureau"),
    coverage_by_client(train_ids, tables["previous_application"], label="previous_application"),
    coverage_by_client(train_ids, tables["POS_CASH_balance"], label="POS_CASH_balance"),
    coverage_by_client(train_ids, tables["installments_payments"], label="installments_payments"),
    coverage_by_client(train_ids, tables["credit_card_balance"], label="credit_card_balance"),
]

# bureau_balance: indirect coverage via bureau.SK_ID_BUREAU
bureau_ids_with_balance = set(tables["bureau_balance"]["SK_ID_BUREAU"].unique())
bureau_curr_with_balance = tables["bureau"][
    tables["bureau"]["SK_ID_BUREAU"].isin(bureau_ids_with_balance)
]["SK_ID_CURR"].unique()
train_set = set(train_ids)
n_train = len(train_set)
covered_bb = set(bureau_curr_with_balance) & train_set
bb_counts = (
    tables["bureau"]
    .loc[tables["bureau"]["SK_ID_BUREAU"].isin(bureau_ids_with_balance)]
    .groupby("SK_ID_CURR")["SK_ID_BUREAU"]
    .nunique()
)
bb_counts = bb_counts.loc[bb_counts.index.isin(covered_bb)]

coverage_rows.append(
    {
        "domain": "bureau_balance (via bureau)",
        "total_clients": n_train,
        "covered_clients": len(covered_bb),
        "%coverage": round(100 * len(covered_bb) / n_train, 2),
        "rows_min": int(bb_counts.min()) if len(covered_bb) else 0,
        "rows_median": float(bb_counts.median()) if len(covered_bb) else 0.0,
        "rows_mean": round(float(bb_counts.mean()), 2) if len(covered_bb) else 0.0,
        "rows_max": int(bb_counts.max()) if len(covered_bb) else 0,
    }
)

coverage_df = pd.DataFrame(coverage_rows)
coverage_df

,domain,total_clients,covered_clients,%coverage,rows_min,rows_median,rows_mean,rows_max
0,bureau,307511,263491,85.69,1,4.0,5.56,116
1,previous_application,307511,291057,94.65,1,4.0,4.86,73
2,POS_CASH_balance,307511,289444,94.12,1,22.0,29.52,295
3,installments_payments,307511,291643,94.84,1,25.0,39.75,372
4,credit_card_balance,307511,86905,28.26,1,22.0,37.14,192
5,bureau_balance (via bureau),307511,92231,29.99,1,5.0,5.68,116


### Column interpretation

| Column | Meaning |
|--------|---------|
| `domain` | Dataset name. |
| `total_clients` | All train clients (`SK_ID_CURR` in `application_train`). Denominator for coverage. |
| `covered_clients` | Train clients with at least one related row in that domain. |
| `%coverage` | Share of train clients with any data in that domain. |
| `rows_min` | Among covered clients only: fewest related records for any single client. Usually `1`. |
| `rows_median` | Typical covered client — robust to outliers. |
| `rows_mean` | Average related records per covered client — pulled up by heavy users. |
| `rows_max` | Most extreme covered client in that domain. |

Intepretation example: `installments_payments` at median 25 means a typical covered client has ~25 installment rows to aggregate in the feature pipeline (`src/features/`).

---

## 4. Train vs test: row counts and column alignment

In [8]:
def compare_train_test_columns(train_df: pd.DataFrame, test_df: pd.DataFrame) -> dict:
    train_cols = set(train_df.columns)
    test_cols = set(test_df.columns)
    return {
        "train_rows": train_df.shape[0],
        "test_rows": test_df.shape[0],
        "train_columns": train_df.shape[1],
        "test_columns": test_df.shape[1],
        "shared_columns": len(train_cols & test_cols),
        "train_only_columns": sorted(train_cols - test_cols),
        "test_only_columns": sorted(test_cols - train_cols),
    }


train_test = compare_train_test_columns(
    tables["application_train"], tables["application_test"]
)

pd.DataFrame(
    [
        {"metric": "train rows", "value": train_test["train_rows"]},
        {"metric": "test rows", "value": train_test["test_rows"]},
        {"metric": "train columns", "value": train_test["train_columns"]},
        {"metric": "test columns", "value": train_test["test_columns"]},
        {"metric": "shared columns", "value": train_test["shared_columns"]},
        {"metric": "train-only columns", "value": ", ".join(train_test["train_only_columns"])},
        {"metric": "test-only columns", "value": ", ".join(train_test["test_only_columns"]) or "—"},
    ]
)

,metric,value
0,train rows,307511
1,test rows,48744
2,train columns,122
3,test columns,121
4,shared columns,121
5,train-only columns,TARGET
6,test-only columns,—


`TARGET` appears only in `application_train` (train-only column). The project uses internal cross-validation on train; Kaggle test submission is out of scope.

---

## 5. Implications for later EDA

- **One-to-many child tables** (bureau, previous applications, installments, POS/cash, credit card) must be aggregated to client level before modeling (done in `src/features/`).
- **Partial coverage** — not every train client has rows in every auxiliary table; engineered features from those domains will be missing for uncovered clients and need imputation or null-aware handling.
- **Train/test schema** — shared application columns align; only `TARGET` differs. Feature engineering pipelines should use the same transforms on both splits.
- **Column-level patterns** (missingness, distributions, correlations) belong in notebooks `01`–`06`, not here.

## Summary of dataset overview

_Executed successfully on 2026-07-14 (all 9 expected CSVs found in `data/`)._

- **Tables loaded:** 9 CSVs spanning ~58.9M total rows. Largest tables: `bureau_balance` (27.3M), `installments_payments` (13.6M), `POS_CASH_balance` (10.0M). Application tables: **307,511 train** and **48,744 test** clients.
- **Relational pattern:** `application_*` → `bureau` / `previous_application` (one-to-many via `SK_ID_CURR`); `bureau` → `bureau_balance` (via `SK_ID_BUREAU`); `previous_application` → installments, POS/cash, and credit card tables (one-to-many via `SK_ID_PREV`).
- **Coverage (train clients):** Highest — `installments_payments` (94.8%), `previous_application` (94.7%), `POS_CASH_balance` (94.1%). Lowest — `credit_card_balance` (28.3%), `bureau_balance` via bureau (30.0%). `bureau` direct coverage: 85.7%.
- **Multiplicity:** Covered clients typically have several bureau/previous-app records (median ~4) and many installment/POS rows (median ~22–25 per client).
- **Train vs test:** 121 shared columns; **only `TARGET` is train-only**. Schemas align for feature engineering.
- **Next steps:** Deep column analysis in notebooks `01`–`06`; aggregation of one-to-many child tables in the feature pipeline (`src/features/`); protected-attribute audit in `07`.